In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.notebook import tqdm

In [2]:
tf = transforms.ToTensor()
train_loader = DataLoader(datasets.MNIST('/tmp/mnist', train=True,  download=True, transform=tf), batch_size=1024, shuffle=True)
test_loader  = DataLoader(datasets.MNIST('/tmp/mnist', train=False, download=True, transform=tf), batch_size=1024)

In [3]:
device = 'cuda:0'

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 512), nn.SiLU(),
    nn.Linear(512, 512), nn.SiLU(),
    nn.Linear(512, 512), nn.SiLU(),
    nn.Linear(512, 10),
).to(device)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)

### Training

In [4]:
for epoch in range(5):
    model.train()
    total_loss = 0
    for x, y in tqdm(train_loader, desc=f'Epoch {epoch+1}'):
        x, y = x.to(device), y.to(device)
        loss = F.cross_entropy(model(x), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}  loss={total_loss/len(train_loader):.4f}')

Epoch 1:   0%|          | 0/59 [00:00<?, ?it/s]

Epoch 1  loss=0.7151


Epoch 2:   0%|          | 0/59 [00:00<?, ?it/s]

Epoch 2  loss=0.2182


Epoch 3:   0%|          | 0/59 [00:00<?, ?it/s]

Epoch 3  loss=0.1541


Epoch 4:   0%|          | 0/59 [00:00<?, ?it/s]

Epoch 4  loss=0.1192


Epoch 5:   0%|          | 0/59 [00:00<?, ?it/s]

Epoch 5  loss=0.0999


### Evaluation

In [5]:
model.eval()
correct = total = 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total += len(y)
print(f'Test accuracy: {correct/total:.4f}')

Test accuracy: 0.9647
